# 01 - Ingesta Parquet a Snowflake RAW

Descarga los Parquets de NYC TLC (Yellow y Green, 2015-2025) **mes a mes** y los carga al esquema `RAW` de Snowflake.

## Imports y configuración


In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import time
import urllib.request
import snowflake.connector

# Credenciales
SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]
SNOWFLAKE_ROLE      = os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN")
SCHEMA_RAW          = os.environ.get("SNOWFLAKE_SCHEMA_RAW", "RAW")

# Parametros
START_YEAR  = int(os.environ.get("START_YEAR", "2015"))
END_YEAR    = int(os.environ.get("END_YEAR", "2025"))
SERVICES    = os.environ.get("SERVICES", "yellow,green").split(",")
CHUNK_SIZE  = os.environ.get("CHUNK_SIZE", "month")
RUN_ID      = os.environ.get("RUN_ID", f"run_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}")
BASE_URL    = os.environ.get("PARQUET_BASE_URL",
              "https://d37ci6vzurychx.cloudfront.net/trip-data")

# Flags
VALIDATE_NULLS      = os.environ.get("VALIDATE_NULLS", "true").lower() == "true"
VALIDATE_RANGES     = os.environ.get("VALIDATE_RANGES", "true").lower() == "true"
VALIDATE_TIMESTAMPS = os.environ.get("VALIDATE_TIMESTAMPS", "true").lower() == "true"

DATA_DIR = "/tmp/parquet_data"
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Ingesta: {SERVICES} | {START_YEAR}-{END_YEAR} | chunk={CHUNK_SIZE}")
print(f"Validaciones: nulls={VALIDATE_NULLS}, ranges={VALIDATE_RANGES}, ts={VALIDATE_TIMESTAMPS}")
print(f"RUN_ID: {RUN_ID}")

spark = SparkSession.builder \
    .appName("P3_Ingesta_Raw") \
    .config("spark.jars.packages",
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4,"
            "net.snowflake:snowflake-jdbc:3.16.1") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

sf_options = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER,
    "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE,
    "sfSchema": SCHEMA_RAW,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE,
    "sfRole": SNOWFLAKE_ROLE,
}

print("Opciones de conexion Spark-Snowflake configuradas.")

Ingesta: ['yellow', 'green'] | 2015-2025 | chunk=month
Validaciones: nulls=True, ranges=True, ts=True
RUN_ID: run_001
Spark version: 3.5.0
Opciones de conexion Spark-Snowflake configuradas.


## Setup de Snowflake

Crea warehouse, database y schemas si no existen. Idempotente con `IF NOT EXISTS`.

In [2]:
conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD,
    role=SNOWFLAKE_ROLE,
)
cursor = conn.cursor()

cursor.execute(f"""CREATE WAREHOUSE IF NOT EXISTS {SNOWFLAKE_WAREHOUSE}
    WITH WAREHOUSE_SIZE='XSMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE""")
print(f"Warehouse {SNOWFLAKE_WAREHOUSE}: OK")

cursor.execute(f"CREATE DATABASE IF NOT EXISTS {SNOWFLAKE_DATABASE}")
print(f"Database {SNOWFLAKE_DATABASE}: OK")

cursor.execute(f"CREATE SCHEMA IF NOT EXISTS {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}")
cursor.execute(f"CREATE SCHEMA IF NOT EXISTS {SNOWFLAKE_DATABASE}.ANALYTICS")
print(f"Schemas {SCHEMA_RAW} y ANALYTICS: OK")

cursor.execute(f"SHOW SCHEMAS IN DATABASE {SNOWFLAKE_DATABASE}")
for row in cursor:
    print(f"  {row[1]}")

conn.close()
print("\nSnowflake setup completo.")

Warehouse COMPUTE_WH: OK
Database NYC_TLC: OK
Schemas RAW y ANALYTICS: OK
  ANALYTICS
  INFORMATION_SCHEMA
  PUBLIC
  RAW

Snowflake setup completo.


## Funciones auxiliares

In [7]:
from tqdm.notebook import tqdm as tqdm_notebook
import shutil

def get_snowflake_connection():
    return snowflake.connector.connect(
        account=SNOWFLAKE_ACCOUNT,
        user=SNOWFLAKE_USER,
        password=SNOWFLAKE_PASSWORD,
        database=SNOWFLAKE_DATABASE,
        schema=SCHEMA_RAW,
        warehouse=SNOWFLAKE_WAREHOUSE,
        role=SNOWFLAKE_ROLE,
    )


def download_parquet(service: str, year: int, month: int, step_bar=None):
    filename = f"{service}_tripdata_{year}-{month:02d}.parquet"
    url = f"{BASE_URL}/{filename}"
    local_path = f"{DATA_DIR}/{filename}"

    try:
        req = urllib.request.urlopen(url)
        total_size = int(req.headers.get("Content-Length", 0))

        with open(local_path, "wb") as f:
            downloaded = 0
            block_size = 1024 * 256  # 256 KB
            dl_bar = tqdm_notebook(
                total=total_size, unit="B", unit_scale=True, unit_divisor=1024,
                desc=f"    {filename}", leave=False
            )
            while True:
                chunk = req.read(block_size)
                if not chunk:
                    break
                f.write(chunk)
                downloaded += len(chunk)
                dl_bar.update(len(chunk))
            dl_bar.close()

        if step_bar:
            step_bar.update(1)
        return local_path
    except Exception as e:
        if step_bar:
            step_bar.update(1)
        return None


def standardize_columns(df, service: str, year: int, month: int, source_path: str):
    """
    Estandariza columnas entre Yellow y Green:
    - Yellow: tpep_pickup/dropoff_datetime -> pickup/dropoff_datetime
    - Green:  lpep_pickup/dropoff_datetime -> pickup/dropoff_datetime
    - Cast a timestamp
    - Agrega metadatos
    """
    cols = [c.lower() for c in df.columns]
    df = df.toDF(*cols)

    if "tpep_pickup_datetime" in cols:
        df = df.withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
               .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
    elif "lpep_pickup_datetime" in cols:
        df = df.withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
               .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")

    df = df.withColumn("pickup_datetime", F.col("pickup_datetime").cast("timestamp")) \
           .withColumn("dropoff_datetime", F.col("dropoff_datetime").cast("timestamp"))

    df = df.withColumn("service_type", F.lit(service)) \
           .withColumn("run_id", F.lit(RUN_ID)) \
           .withColumn("source_year", F.lit(year)) \
           .withColumn("source_month", F.lit(month)) \
           .withColumn("source_path", F.lit(source_path)) \
           .withColumn("ingested_at_utc", F.current_timestamp())

    return df


def compute_quality_stats(df, service: str, year: int, month: int):
    total = df.count()
    stats = {"total_count": total, "null_timestamps": 0, "bad_timestamps": 0, "range_violations": 0}

    if VALIDATE_NULLS:
        stats["null_timestamps"] = df.filter(
            F.col("pickup_datetime").isNull() | F.col("dropoff_datetime").isNull()
        ).count()

    if VALIDATE_TIMESTAMPS:
        stats["bad_timestamps"] = df.filter(
            F.col("pickup_datetime") > F.col("dropoff_datetime")
        ).count()

    if VALIDATE_RANGES:
        range_issues = 0
        if "trip_distance" in df.columns:
            range_issues += df.filter(F.col("trip_distance") < 0).count()
        if "total_amount" in df.columns:
            range_issues += df.filter(F.col("total_amount") < 0).count()
        stats["range_violations"] = range_issues

    stats["issues_total"] = stats["null_timestamps"] + stats["bad_timestamps"] + stats["range_violations"]

    return stats


def delete_existing_partition(service: str, year: int, month: int):
    table_name = f"TRIPS_{service.upper()}"
    conn = get_snowflake_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(f"""
            DELETE FROM {SCHEMA_RAW}.{table_name}
            WHERE source_year = {year} AND source_month = {month}
        """)
        return cursor.rowcount
    except Exception:
        return 0
    finally:
        conn.close()


def write_to_snowflake_raw(df, service: str, year: int, month: int):
    table_name = f"TRIPS_{service.upper()}"

    start_time = time.time()

    deleted = delete_existing_partition(service, year, month)

    df.write.format("net.snowflake.spark.snowflake") \
        .options(**sf_options) \
        .option("dbtable", table_name) \
        .option("column_mapping", "name") \
        .option("column_mismatch_behavior", "ignore") \
        .mode("append") \
        .save()

    elapsed = round(time.time() - start_time, 2)
    return elapsed, deleted


print("Funciones auxiliares cargadas.")

Funciones auxiliares cargadas.


## Ingesta mes a mes

In [4]:
audit_log = []

STEPS = ["Descarga", "Lectura", "Estandarizar", "Calidad", "Escritura SF"]

chunks = [(s, y, m) for s in SERVICES for y in range(START_YEAR, END_YEAR + 1) for m in range(1, 13)]

pbar = tqdm_notebook(chunks, desc="Ingesta RAW", unit="chunk")

for service, year, month in pbar:
    pbar.set_postfix_str(f"{service} {year}-{month:02d}")

    source_url = f"{BASE_URL}/{service}_tripdata_{year}-{month:02d}.parquet"

    step_bar = tqdm_notebook(
        total=len(STEPS), desc=f"  {service} {year}-{month:02d} | {STEPS[0]}",
        leave=False, bar_format="{desc} {bar} {n_fmt}/{total_fmt} [{elapsed}]"
    )

    # Paso 1: Descarga
    local_path = download_parquet(service, year, month, step_bar)

    if local_path is None:
        step_bar.close()
        audit_log.append({
            "run_id": RUN_ID, "service_type": service,
            "source_year": year, "source_month": month,
            "status": "missing", "row_count": 0,
            "issues_found": 0, "load_time_sec": 0.0,
            "source_path": source_url
        })
        continue

    try:
        # Paso 2: Lectura Parquet
        step_bar.set_description(f"  {service} {year}-{month:02d} | {STEPS[1]}")
        df = spark.read.parquet(local_path)
        step_bar.update(1)

        # Paso 3: Estandarizar columnas + metadatos
        step_bar.set_description(f"  {service} {year}-{month:02d} | {STEPS[2]}")
        df = standardize_columns(df, service, year, month, source_url)
        step_bar.update(1)

        # Paso 4: Calidad (reporte)
        step_bar.set_description(f"  {service} {year}-{month:02d} | {STEPS[3]}")
        quality = compute_quality_stats(df, service, year, month)
        step_bar.update(1)

        # Paso 5: Escritura a Snowflake
        step_bar.set_description(f"  {service} {year}-{month:02d} | {STEPS[4]}")
        elapsed, deleted = write_to_snowflake_raw(df, service, year, month)
        step_bar.update(1)
        step_bar.close()

        if deleted > 0:
            print(f"  IDEMPOTENCIA [{service} {year}-{month:02d}]: borradas {deleted:,} filas previas antes de reinsertar")

        audit_log.append({
            "run_id": RUN_ID, "service_type": service,
            "source_year": year, "source_month": month,
            "status": "ok",
            "row_count": quality["total_count"],
            "issues_found": quality["issues_total"],
            "null_timestamps": quality["null_timestamps"],
            "bad_timestamps": quality["bad_timestamps"],
            "range_violations": quality["range_violations"],
            "load_time_sec": elapsed,
            "deleted_prev": deleted,
            "source_path": source_url
        })
        pbar.set_postfix_str(f"{service} {year}-{month:02d} | {quality['total_count']:,} filas | {elapsed}s")

    except Exception as e:
        step_bar.close()
        audit_log.append({
            "run_id": RUN_ID, "service_type": service,
            "source_year": year, "source_month": month,
            "status": "failed", "row_count": 0,
            "issues_found": 0, "load_time_sec": 0.0,
            "source_path": source_url, "error": str(e)
        })
        print(f"\n  ERROR [{service} {year}-{month:02d}]: {e}")

    finally:
        if local_path and os.path.exists(local_path):
            os.remove(local_path)

pbar.close()
ok = sum(1 for r in audit_log if r["status"] == "ok")
missing = sum(1 for r in audit_log if r["status"] == "missing")
failed = sum(1 for r in audit_log if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in audit_log)
print(f"\nIngesta finalizada: {ok} ok | {missing} missing | {failed} failed | {total_rows:,} filas totales")

Ingesta RAW:   0%|          | 0/264 [00:00<?, ?chunk/s]

  yellow 2015-01 | Descarga            0/5 [00:00]

    yellow_tripdata_2015-01.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

  IDEMPOTENCIA [yellow 2015-01]: borradas 12,741,035 filas previas antes de reinsertar


  yellow 2015-02 | Descarga            0/5 [00:00]

...

    green_tripdata_2025-11.parquet:   0%|          | 0.00/1.11M [00:00<?, ?B/s]


  ERROR [green 2025-11]: An error occurred while calling o14779.save.
: java.sql.SQLException: Status of query associated with resultSet is FAILED_WITH_ERROR. Number of columns in file (27) does not match that of the corresponding table (26), use file format option error_on_column_count_mismatch=false to ignore this error
  File 'rlol90qDYf/0.CSV.gz', line 2, character 1
  Row 1 starts at line 1, column "TRIPS_GREEN"[27]
  If you would like to continue loading when an error is encountered, use other values such as 'SKIP_FILE' or 'CONTINUE' for the ON_ERROR option. For more information on loading options, please run 'info loading_data' in a SQL client. Results not generated.
	at net.snowflake.client.jdbc.SFAsyncResultSet.getRealResults(SFAsyncResultSet.java:159)
	at net.snowflake.client.jdbc.SFAsyncResultSet.getMetaData(SFAsyncResultSet.java:298)
	at net.snowflake.spark.snowflake.io.StageWriter$.executeCopyIntoTable(StageWriter.scala:568)
	at net.snowflake.spark.snowflake.io.StageWrite

  green 2025-12 | Descarga            0/5 [00:00]

    green_tripdata_2025-12.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]


  ERROR [green 2025-12]: An error occurred while calling o14836.save.
: java.sql.SQLException: Status of query associated with resultSet is FAILED_WITH_ERROR. Number of columns in file (27) does not match that of the corresponding table (26), use file format option error_on_column_count_mismatch=false to ignore this error
  File 'cRzN2uoWui/0.CSV.gz', line 2, character 1
  Row 1 starts at line 1, column "TRIPS_GREEN"[27]
  If you would like to continue loading when an error is encountered, use other values such as 'SKIP_FILE' or 'CONTINUE' for the ON_ERROR option. For more information on loading options, please run 'info loading_data' in a SQL client. Results not generated.
	at net.snowflake.client.jdbc.SFAsyncResultSet.getRealResults(SFAsyncResultSet.java:159)
	at net.snowflake.client.jdbc.SFAsyncResultSet.getMetaData(SFAsyncResultSet.java:298)
	at net.snowflake.spark.snowflake.io.StageWriter$.executeCopyIntoTable(StageWriter.scala:568)
	at net.snowflake.spark.snowflake.io.StageWrite

In [8]:
audit_log = []

STEPS = ["Descarga", "Lectura", "Estandarizar", "Calidad", "Escritura SF"]

chunks = [(s, 2025, m) for s in SERVICES for m in range(1, 13)]

pbar = tqdm_notebook(chunks, desc="Ingesta RAW", unit="chunk")

for service, year, month in pbar:
    pbar.set_postfix_str(f"{service} {year}-{month:02d}")

    source_url = f"{BASE_URL}/{service}_tripdata_{year}-{month:02d}.parquet"

    step_bar = tqdm_notebook(
        total=len(STEPS), desc=f"  {service} {year}-{month:02d} | {STEPS[0]}",
        leave=False, bar_format="{desc} {bar} {n_fmt}/{total_fmt} [{elapsed}]"
    )

    # Paso 1: Descarga
    local_path = download_parquet(service, year, month, step_bar)

    if local_path is None:
        step_bar.close()
        audit_log.append({
            "run_id": RUN_ID, "service_type": service,
            "source_year": year, "source_month": month,
            "status": "missing", "row_count": 0,
            "issues_found": 0, "load_time_sec": 0.0,
            "source_path": source_url
        })
        continue

    try:
        # Paso 2: Lectura Parquet
        step_bar.set_description(f"  {service} {year}-{month:02d} | {STEPS[1]}")
        df = spark.read.parquet(local_path)
        step_bar.update(1)

        # Paso 3: Estandarizar columnas + metadatos
        step_bar.set_description(f"  {service} {year}-{month:02d} | {STEPS[2]}")
        df = standardize_columns(df, service, year, month, source_url)
        step_bar.update(1)

        # Paso 4: Calidad (reporte)
        step_bar.set_description(f"  {service} {year}-{month:02d} | {STEPS[3]}")
        quality = compute_quality_stats(df, service, year, month)
        step_bar.update(1)

        # Paso 5: Escritura a Snowflake
        step_bar.set_description(f"  {service} {year}-{month:02d} | {STEPS[4]}")
        elapsed, deleted = write_to_snowflake_raw(df, service, year, month)
        step_bar.update(1)
        step_bar.close()

        if deleted > 0:
            print(f"  IDEMPOTENCIA [{service} {year}-{month:02d}]: borradas {deleted:,} filas previas antes de reinsertar")

        audit_log.append({
            "run_id": RUN_ID, "service_type": service,
            "source_year": year, "source_month": month,
            "status": "ok",
            "row_count": quality["total_count"],
            "issues_found": quality["issues_total"],
            "null_timestamps": quality["null_timestamps"],
            "bad_timestamps": quality["bad_timestamps"],
            "range_violations": quality["range_violations"],
            "load_time_sec": elapsed,
            "deleted_prev": deleted,
            "source_path": source_url
        })
        pbar.set_postfix_str(f"{service} {year}-{month:02d} | {quality['total_count']:,} filas | {elapsed}s")

    except Exception as e:
        step_bar.close()
        audit_log.append({
            "run_id": RUN_ID, "service_type": service,
            "source_year": year, "source_month": month,
            "status": "failed", "row_count": 0,
            "issues_found": 0, "load_time_sec": 0.0,
            "source_path": source_url, "error": str(e)
        })
        print(f"\n  ERROR [{service} {year}-{month:02d}]: {e}")

    finally:
        if local_path and os.path.exists(local_path):
            os.remove(local_path)

pbar.close()
ok = sum(1 for r in audit_log if r["status"] == "ok")
missing = sum(1 for r in audit_log if r["status"] == "missing")
failed = sum(1 for r in audit_log if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in audit_log)
print(f"\nIngesta finalizada: {ok} ok | {missing} missing | {failed} failed | {total_rows:,} filas totales")

Ingesta RAW:   0%|          | 0/24 [00:00<?, ?chunk/s]

  yellow 2025-01 | Descarga            0/5 [00:00]

    yellow_tripdata_2025-01.parquet:   0%|          | 0.00/56.4M [00:00<?, ?B/s]

  yellow 2025-02 | Descarga            0/5 [00:00]

    yellow_tripdata_2025-02.parquet:   0%|          | 0.00/57.5M [00:00<?, ?B/s]

...

  green 2025-11 | Descarga            0/5 [00:00]

    green_tripdata_2025-11.parquet:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

  green 2025-12 | Descarga            0/5 [00:00]

    green_tripdata_2025-12.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]


Ingesta finalizada: 24 ok | 0 missing | 0 failed | 49,313,977 filas totales


## Registra auditoria en Snowflake

In [9]:
audit_df = spark.createDataFrame(audit_log)

audit_df.write.format("net.snowflake.spark.snowflake") \
    .options(**sf_options) \
    .option("dbtable", "INGESTION_AUDIT") \
    .mode("overwrite") \
    .save()

print(f"Auditoria registrada en {SCHEMA_RAW}.INGESTION_AUDIT ({len(audit_log)} registros)")

Auditoria registrada en RAW.INGESTION_AUDIT (24 registros)


## Matriz de cobertura y resumen

In [10]:
import pandas as pd

audit_pd = pd.DataFrame(audit_log)

print("=" * 60)
print("MATRIZ DE COBERTURA 2015-2025")
print("=" * 60)
pivot = audit_pd.pivot_table(
    index=["source_year", "source_month"],
    columns="service_type",
    values="status",
    aggfunc="first"
)
print(pivot.to_string())

print(f"\n{'='*60}")
print("RESUMEN POR SERVICIO")
print("=" * 60)
summary = audit_pd.groupby(["service_type", "status"]).agg(
    meses=("source_month", "count"),
    total_filas=("row_count", "sum"),
    issues=("issues_found", "sum"),
    tiempo_total_seg=("load_time_sec", "sum")
)
print(summary.to_string())

print(f"\n{'='*60}")
print("DETALLE DE ISSUES DE CALIDAD (solo chunks con problemas)")
print("=" * 60)
ok_rows = audit_pd[audit_pd["status"] == "ok"]
if "null_timestamps" in ok_rows.columns:
    issues = ok_rows[ok_rows["issues_found"] > 0][
        ["service_type", "source_year", "source_month", "row_count",
         "null_timestamps", "bad_timestamps", "range_violations"]
    ]
    if len(issues) > 0:
        print(issues.to_string(index=False))
    else:
        print("Ningun chunk con issues de calidad.")
else:
    print("Validaciones desactivadas.")

MATRIZ DE COBERTURA 2015-2025
service_type             green yellow
source_year source_month             
2025        1               ok     ok
            2               ok     ok
            3               ok     ok
            4               ok     ok
            5               ok     ok
            6               ok     ok
            7               ok     ok
            8               ok     ok
            9               ok     ok
            10              ok     ok
            11              ok     ok
            12              ok     ok

RESUMEN POR SERVICIO
                     meses  total_filas  issues  tiempo_total_seg
service_type status                                              
green        ok         12       591375    3118             91.38
yellow       ok         12     48722602  975956            468.23

DETALLE DE ISSUES DE CALIDAD (solo chunks con problemas)
service_type  source_year  source_month  row_count  null_timestamps  bad_timestamps  range_vio